# Phân loại chủ đề bài viết tiếng Việt trên kho ngữ liệu VNTC

Báo cáo cuối kỳ môn Xử lý ngôn ngữ tự nhiên. Lớp N01, nhóm 25.

Notebook này gộp cả hai nhánh thực nghiệm vào một tệp duy nhất:

Nhánh cổ điển gồm TF-IDF kết hợp Naive Bayes, Logistic Regression và Linear SVM.
Nhánh hiện đại là tinh chỉnh mô hình PhoBERT.

Bạn chọn chạy nhánh nào bằng hai công tắc ở Phần 0. Dữ liệu được tải tự động
từ GitHub, không cần tải tệp lên thủ công.

## Mức độ kiểm chứng của từng phần

Phần 1 đến Phần 8 và Phần 10 đến Phần 11 đã được nhóm em chạy thử trên toàn bộ
dữ liệu và cho kết quả sạch.

Phần 9 là phần PhoBERT. Nhóm em KHÔNG chạy thử được phần này vì máy dùng để soạn
notebook không có GPU và không tải được mô hình từ kho Hugging Face. Code phần 9
được viết theo tài liệu chính thức của thư viện transformers và của PhoBERT, nhưng
chưa qua thực nghiệm. Nếu chạy báo lỗi, đó là điều nằm trong dự liệu; hãy gửi lại
nguyên văn thông báo lỗi để sửa.

## Thứ tự chạy đề xuất

Lần một: đặt CHAY_CO_DIEN = True và CHAY_PHOBERT = False, chạy hết một lượt.
Bước này khoảng 15 phút và cho toàn bộ kết quả nhánh cổ điển.

Lần hai: đặt CHAY_CO_DIEN = False và CHAY_PHOBERT = True. Phần tách từ sẽ tự đọc
lại tệp đã lưu ở lần một nên không phải làm lại từ đầu.

## Phần 0. Công tắc cấu hình

In [ ]:
# ================== CÁC CÔNG TẮC CHÍNH ==================

# Chạy nhánh cổ điển (TF-IDF kết hợp Naive Bayes, Logistic Regression, Linear SVM)
CHAY_CO_DIEN = True

# Chạy nhánh PhoBERT. Bắt buộc phải có GPU.
CHAY_PHOBERT = False

# Chạy kiểm định chéo 5 lớp. Phần này fit lại TF-IDF 15 lần nên khá lâu.
CHAY_KIEM_DINH_CHEO = True

# Chế độ chạy thử nhanh. Khi nộp bài thì để False.
MAU_THU = False
SO_MAU_MOI_LOP = 300          # chỉ có tác dụng khi MAU_THU = True

# ================== THAM SỐ PhoBERT ==================

MO_HINH_GOC = "vinai/phobert-base"   # có thể đổi thành vinai/phobert-base-v2
DO_DAI_TOI_DA = 256                  # giới hạn cứng của PhoBERT, không tăng được
BATCH_SIZE = 16                      # hạ xuống 8 nếu báo lỗi hết bộ nhớ GPU
GOP_DAO_HAM = 2                      # batch hiệu dụng = BATCH_SIZE nhân GOP_DAO_HAM
SO_EPOCH = 3
TOC_DO_HOC = 2e-5
KIEU_CAT = "head"                    # "head" lấy phần đầu, "head_tail" lấy đầu cộng cuối

HAT_GIONG = 42

print("Cấu hình đã chọn:")
print("   Nhánh cổ điển   :", CHAY_CO_DIEN)
print("   Nhánh PhoBERT   :", CHAY_PHOBERT)
print("   Kiểm định chéo  :", CHAY_KIEM_DINH_CHEO)
print("   Chế độ chạy thử :", MAU_THU)

## Phần 1. Cài đặt thư viện và kiểm tra môi trường

In [ ]:
# unar dùng để giải nén tệp .rar của bộ dữ liệu (unzip không mở được định dạng này).
# pyvi dùng để tách từ tiếng Việt.
!apt-get -qq install -y unar > /dev/null 2>&1
!pip install -q pyvi joblib

import sys, os, re, csv, json, time, math, random, hashlib, unicodedata, collections, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
import sklearn
import matplotlib.pyplot as plt

from pyvi import ViTokenizer

random.seed(HAT_GIONG)
np.random.seed(HAT_GIONG)

print("Python      :", sys.version.split()[0])
print("numpy       :", np.__version__)
print("pandas      :", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("matplotlib  :", matplotlib.__version__)
print("unar        :", "có" if os.system("which unar > /dev/null") == 0 else "KHÔNG CÓ")
print("Số CPU      :", os.cpu_count())
print("Thử tách từ :", ViTokenizer.tokenize("Bộ Giáo dục và Đào tạo vừa công bố phương án thi"))

In [ ]:
# Cài thư viện cho nhánh PhoBERT và kiểm tra GPU.
# Cell này chỉ làm gì đó khi CHAY_PHOBERT = True.

if CHAY_PHOBERT:
    !pip install -q transformers torch

    import torch
    print("PyTorch     :", torch.__version__)
    print("Bản CUDA mà PyTorch được dựng theo:", torch.version.cuda)
    print("Thấy GPU    :", torch.cuda.is_available())

    if torch.cuda.is_available():
        ten_gpu = torch.cuda.get_device_name(0)
        kha_nang = torch.cuda.get_device_capability(0)
        bo_nho = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        print("Tên GPU     :", ten_gpu)
        print("Compute capability:", kha_nang)
        print("VRAM        :", round(bo_nho, 1), "GB")
        print()

        # CẢNH BÁO RIÊNG CHO CARD DÒNG RTX 50 (kiến trúc Blackwell).
        # Các card này có compute capability sm_120. Những bản PyTorch dựng theo
        # CUDA 12.1 hoặc 12.4 KHÔNG chứa nhân biên dịch cho kiến trúc này.
        # Triệu chứng rất dễ gây hiểu nhầm: dòng "Thấy GPU" ở trên vẫn in ra True,
        # nhưng tới lúc chạy thật thì báo lỗi không tìm thấy nhân thực thi.
        if kha_nang[0] >= 12:
            print("LƯU Ý: GPU này thuộc kiến trúc Blackwell (sm_120).")
            print("Nếu tới Phần 9 báo lỗi dạng no kernel image is available,")
            print("hãy cài lại PyTorch theo bản CUDA 12.8 trở lên bằng lệnh:")
            print("   pip install --force-reinstall torch --index-url https://download.pytorch.org/whl/cu128")
            print()

        # Thử một phép nhân ma trận nhỏ để xác nhận GPU chạy được thật
        try:
            a = torch.randn(64, 64, device="cuda")
            b = a @ a
            torch.cuda.synchronize()
            print("Thử tính trên GPU: THÀNH CÔNG, GPU dùng được")
        except Exception as e:
            print("Thử tính trên GPU: THẤT BẠI")
            print("   ", type(e).__name__, str(e)[:200])
            print("   Đây gần như chắc chắn là vấn đề phiên bản PyTorch nêu ở trên.")

        # Gợi ý batch size theo dung lượng VRAM
        if bo_nho < 9:
            print()
            print("Với VRAM dưới 9 GB, nên giữ BATCH_SIZE = 16 hoặc hạ xuống 8.")
    else:
        print("KHÔNG THẤY GPU. Nhánh PhoBERT sẽ chạy rất chậm trên CPU.")
        print("Trên Colab, vào Runtime rồi Change runtime type rồi chọn GPU.")
else:
    print("Bỏ qua cell này vì CHAY_PHOBERT = False")

## Phần 2. Tải và đọc dữ liệu VNTC

Hai điểm kỹ thuật dễ vấp ở phần này.

Một, bộ dữ liệu được nén ở định dạng .rar chứ không phải .zip, nên phải dùng unar.

Hai, các tệp văn bản được lưu bằng bảng mã UTF-16 Little Endian có BOM, xuống dòng
kiểu CRLF. Nếu mở bằng encoding mặc định là UTF-8 thì sẽ ra ký tự rác hoặc báo lỗi.
Bắt buộc phải ghi rõ encoding utf-16.

In [ ]:
THU_MUC = "vntc_ext"
TEP_CACHE = "du_lieu_da_tach_tu.csv"

# Nếu lần chạy trước đã lưu tệp đã tách từ thì bỏ qua hẳn việc tải và giải nén,
# tiết kiệm khoảng 160 MB tải về cùng vài phút giải nén.
DA_CO_CACHE = os.path.exists(TEP_CACHE)

if DA_CO_CACHE:
    print("Đã có", TEP_CACHE, "từ lần chạy trước.")
    print("Bỏ qua bước tải và giải nén dữ liệu gốc.")
else:
    # Tải kho dữ liệu từ GitHub. Kích thước khoảng 160 MB.
    if not os.path.exists("VNTC"):
        !git clone --depth 1 -q https://github.com/duyvuleo/VNTC.git

    # Giải nén hai tệp .rar của phiên bản 10 chủ đề
    if not os.path.exists(os.path.join(THU_MUC, "Train_Full")):
        os.makedirs(THU_MUC, exist_ok=True)
        !unar -q -o vntc_ext VNTC/Data/10Topics/Ver1.1/Train_Full.rar > /dev/null
        !unar -q -o vntc_ext VNTC/Data/10Topics/Ver1.1/Test_Full.rar > /dev/null

    print("Các thư mục chủ đề trong Train_Full:")
    for t in sorted(os.listdir(os.path.join(THU_MUC, "Train_Full"))):
        print("   ", t)

In [ ]:
# Tên hiển thị có dấu, dùng cho bảng biểu và biểu đồ.
# Tên thư mục trong bộ dữ liệu là không dấu nên phải ánh xạ lại.
TEN_HIEN_THI = {
    "Chinh tri Xa hoi": "Chính trị Xã hội",
    "Doi song": "Đời sống",
    "Khoa hoc": "Khoa học",
    "Kinh doanh": "Kinh doanh",
    "Phap luat": "Pháp luật",
    "Suc khoe": "Sức khỏe",
    "The gioi": "Thế giới",
    "The thao": "Thể thao",
    "Van hoa": "Văn hóa",
    "Vi tinh": "Vi tính",
}

# Bốn tòa soạn có trong bộ dữ liệu. Mã tòa soạn nằm ở TÊN TỆP, không nằm trong nội dung.
MA_TOA_SOAN = {"VNE": "VnExpress", "TT": "Tuổi Trẻ", "TN": "Thanh Niên", "NLD": "Người Lao Động"}

# Bẫy cần tránh: TT vừa là mã chủ đề Thể thao vừa là mã báo Tuổi Trẻ.
# Vì vậy phải lấy đúng token thứ hai trong tên tệp, không được tìm chuỗi bừa bãi.
MAU_TEN_TEP = re.compile(r"^([A-Za-z]+)_\s*([A-Za-z]+)_")


def doc_mot_tep(duong_dan):
    """Đọc một tệp văn bản của VNTC và trả về chuỗi đã chuẩn hóa."""
    with open(duong_dan, "rb") as f:
        raw = f.read()
    # Bắt buộc giải mã bằng utf-16, không được để mặc định
    van_ban = raw.decode("utf-16", errors="replace")
    # Chuẩn hóa Unicode về dạng tổ hợp sẵn NFC. Tiếng Việt có thể được lưu ở hai
    # dạng khác nhau mà nhìn bằng mắt thì giống hệt; nếu không chuẩn hóa thì cùng
    # một chữ lại sinh ra hai đặc trưng khác nhau.
    van_ban = unicodedata.normalize("NFC", van_ban)
    # Gộp mọi khoảng trắng, tab, xuống dòng thành một dấu cách duy nhất
    return " ".join(van_ban.split())


def quet_toan_bo():
    ban_ghi = []
    for tap in ["Train_Full", "Test_Full"]:
        goc = os.path.join(THU_MUC, tap)
        for chu_de in sorted(os.listdir(goc)):
            thu_muc_lop = os.path.join(goc, chu_de)
            if not os.path.isdir(thu_muc_lop):
                continue
            for ten in sorted(os.listdir(thu_muc_lop)):
                if not ten.lower().endswith(".txt"):
                    continue
                vb = doc_mot_tep(os.path.join(thu_muc_lop, ten))
                m = MAU_TEN_TEP.match(ten)
                ma = m.group(2).upper() if m else "KHAC"
                ban_ghi.append({
                    "tap": "train" if tap == "Train_Full" else "test",
                    "chu_de": chu_de,
                    "ten_tep": ten,
                    "toa_soan": ma if ma in MA_TOA_SOAN else "KHAC",
                    "so_tu": len(vb.split()),
                    "van_ban": vb,
                    # Băm MD5 trên nội dung đã hạ chữ thường, dùng để khử trùng lặp ở cell sau
                    "bam": hashlib.md5(vb.lower().encode("utf-8")).hexdigest(),
                })
    return ban_ghi


if DA_CO_CACHE:
    print("Dùng lại dữ liệu đã tách từ, bỏ qua bước đọc tệp gốc.")
    ban_ghi = None
    tk_trung_lap = None
else:
    t0 = time.time()
    ban_ghi = quet_toan_bo()
    print("Đọc xong", len(ban_ghi), "văn bản trong", round(time.time() - t0, 1), "giây")

    mau = os.path.join(THU_MUC, "Train_Full", "The thao")
    mau = os.path.join(mau, sorted(os.listdir(mau))[0])
    with open(mau, "rb") as f:
        dau = f.read(2)
    print("Hai byte đầu của một tệp mẫu:", dau.hex(),
          "(fffe nghĩa là UTF-16 Little Endian có BOM)")

In [ ]:
# KHỬ TRÙNG LẶP
#
# Bộ dữ liệu gốc có hiện tượng trùng lặp nội dung giữa tập huấn luyện và tập kiểm thử.
# Nếu không xử lý, mô hình sẽ được chấm điểm trên chính những văn bản nó đã nhìn thấy,
# tức là đo khả năng ghi nhớ chứ không phải khả năng khái quát.
#
# Quy tắc áp dụng:
#   1. Nhóm nội dung giống hệt nhau nhưng bị gán HAI nhãn khác nhau thì bỏ CẢ nhóm,
#      vì không có căn cứ nào để biết nhãn nào đúng.
#   2. Nhóm trùng lặp cùng nhãn thì giữ lại một bản, ưu tiên giữ bản nằm ở tập train,
#      để bản sao bên tập test bị loại và không còn rò rỉ.

def khu_trung_lap(ban_ghi):
    nhom = collections.defaultdict(list)
    for i, r in enumerate(ban_ghi):
        nhom[r["bam"]].append(i)

    bo_di = set()
    so_nhom_mau_thuan = 0
    so_tep_mau_thuan = 0
    so_trung_train_test = 0

    for _, chi_so in nhom.items():
        nhan = {ban_ghi[i]["chu_de"] for i in chi_so}
        if len(nhan) > 1:
            so_nhom_mau_thuan += 1
            so_tep_mau_thuan += len(chi_so)
            bo_di.update(chi_so)
            continue
        if len(chi_so) == 1:
            continue
        cac_tap = {ban_ghi[i]["tap"] for i in chi_so}
        if "train" in cac_tap and "test" in cac_tap:
            so_trung_train_test += 1
        giu = None
        for i in chi_so:
            if ban_ghi[i]["tap"] == "train":
                giu = i
                break
        if giu is None:
            giu = chi_so[0]
        bo_di.update(set(chi_so) - {giu})

    con_lai = [r for i, r in enumerate(ban_ghi) if i not in bo_di]
    thong_ke = {
        "tong_ban_dau": len(ban_ghi),
        "so_tep_bi_loai": len(bo_di),
        "so_nhom_nhan_mau_thuan": so_nhom_mau_thuan,
        "so_tep_thuoc_nhom_mau_thuan": so_tep_mau_thuan,
        "so_noi_dung_trung_ca_train_va_test": so_trung_train_test,
        "tong_con_lai": len(con_lai),
    }
    return con_lai, thong_ke


if not DA_CO_CACHE:
    ban_ghi, tk_trung_lap = khu_trung_lap(ban_ghi)
    with open("thong_ke_khu_trung_lap.json", "w", encoding="utf-8") as f:
        json.dump(tk_trung_lap, f, ensure_ascii=False, indent=1)
    print("KẾT QUẢ KHỬ TRÙNG LẶP")
    for k, v in tk_trung_lap.items():
        print("   ", k, "=", v)
else:
    if os.path.exists("thong_ke_khu_trung_lap.json"):
        with open("thong_ke_khu_trung_lap.json", encoding="utf-8") as f:
            tk_trung_lap = json.load(f)
        print("Đọc lại thống kê khử trùng lặp từ lần chạy trước:")
        for k, v in tk_trung_lap.items():
            print("   ", k, "=", v)
    else:
        tk_trung_lap = {}
        print("Không có tệp thống kê khử trùng lặp, bỏ qua.")

## Phần 3. Thống kê mô tả và biểu đồ phân bố nhãn

In [ ]:
# Cài font có dấu tiếng Việt cho biểu đồ.
# Font mặc định của matplotlib trên Colab thiếu một số ký tự tiếng Việt,
# khi đó nhãn trên biểu đồ sẽ hiện thành ô vuông.
!apt-get -qq install -y fonts-dejavu-core > /dev/null 2>&1
try:
    matplotlib.font_manager.fontManager.addfont("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf")
    plt.rcParams["font.family"] = "DejaVu Sans"
    print("Đã đặt font DejaVu Sans cho biểu đồ")
except Exception as e:
    print("Không đặt được font, biểu đồ có thể thiếu dấu:", type(e).__name__)

In [ ]:
if not DA_CO_CACHE:
    df_tk = pd.DataFrame([{k: r[k] for k in ["tap", "chu_de", "toa_soan", "so_tu"]}
                          for r in ban_ghi])
else:
    df_tk = pd.read_csv(TEP_CACHE, usecols=["tap", "chu_de", "toa_soan", "so_tu"])

bang = df_tk.pivot_table(index="chu_de", columns="tap", values="so_tu",
                         aggfunc="count").fillna(0).astype(int)
bang = bang[["train", "test"]]
bang["tong"] = bang["train"] + bang["test"]
bang.loc["TONG"] = bang.sum()
print("BẢNG THỐNG KÊ MÔ TẢ THEO CHỦ ĐỀ")
print(bang.to_string())
print()

print("BẢNG PHÂN BỐ THEO TÒA SOẠN (kiểm tra chéo rò rỉ dữ liệu)")
bang_bao = pd.crosstab(df_tk["chu_de"], df_tk["toa_soan"])
print(bang_bao.to_string())
print()
print("Mỗi chủ đề đều có bài từ bao nhiêu nguồn báo:",
      (bang_bao > 0).sum(axis=1).unique(), "nguồn")
print()

print("ĐỘ DÀI VĂN BẢN (đếm theo khoảng trắng, chưa tách từ)")
do_dai = df_tk.groupby("tap")["so_tu"].agg(
    trung_binh="mean", trung_vi="median", nho_nhat="min", lon_nhat="max").round(1)
print(do_dai.to_string())

In [ ]:
sap_xep = bang.drop("TONG").sort_values("tong", ascending=False)
nhan_ve = [TEN_HIEN_THI.get(i, i) for i in sap_xep.index]

fig, ax = plt.subplots(1, 2, figsize=(15, 5))

vi_tri = np.arange(len(sap_xep))
ax[0].bar(vi_tri - 0.2, sap_xep["train"], width=0.4, label="Train")
ax[0].bar(vi_tri + 0.2, sap_xep["test"], width=0.4, label="Test")
ax[0].set_xticks(vi_tri)
ax[0].set_xticklabels(nhan_ve, rotation=45, ha="right")
ax[0].set_ylabel("Số văn bản")
ax[0].set_title("Phân bố nhãn theo tập")
ax[0].legend()

ax[1].barh(nhan_ve[::-1], sap_xep["tong"][::-1])
ax[1].set_xlabel("Tổng số văn bản")
ax[1].set_title("Tổng số văn bản mỗi chủ đề")

plt.tight_layout()
plt.savefig("bieu_do_phan_bo_nhan.png", dpi=150, bbox_inches="tight")
plt.show()
print("Đã lưu bieu_do_phan_bo_nhan.png để đưa vào báo cáo")

ty_le = sap_xep["tong"].max() / sap_xep["tong"].min()
print("Mức lệch giữa lớp lớn nhất và lớp nhỏ nhất:", round(ty_le, 2), "lần")
print("Vì lệch như vậy nên chỉ số chính của báo cáo là macro-F1, không phải accuracy.")

## Phần 4. Tiền xử lý và tách từ tiếng Việt

Bước chuẩn hóa Unicode và gộp khoảng trắng đã làm ở Phần 2. Còn lại ở đây là bước
tách từ, tức biến chuỗi âm tiết thành chuỗi từ ghép nối bằng gạch dưới.

Ví dụ: chuỗi "học sinh giỏi" trở thành "học_sinh giỏi".

Đây là bước bắt buộc với tiếng Việt vì đơn vị được ngăn cách bằng khoảng trắng là
ÂM TIẾT chứ không phải TỪ. Nếu tách theo khoảng trắng như tiếng Anh thì cụm hai âm
tiết bị cắt rời và mất nghĩa.

Cả hai nhánh dùng CHUNG kết quả tách từ này. Đó là chủ ý: nếu hai nhánh dùng tiền
xử lý khác nhau thì lúc so sánh không biết chênh lệch đến từ mô hình hay đến từ
tiền xử lý.

Đây là bước lâu nhất trong cả notebook, khoảng 7 tới 8 phút cho toàn bộ dữ liệu.
Kết quả được lưu ra tệp CSV nên lần chạy sau sẽ tự đọc lại và bỏ qua bước này.

In [ ]:
def tien_xu_ly(van_ban):
    """Chuỗi tiền xử lý dùng chung cho cả lúc huấn luyện và lúc dự đoán.
    Viết thành một hàm duy nhất để tránh lệch giữa hai nơi."""
    van_ban = unicodedata.normalize("NFC", van_ban)
    van_ban = " ".join(van_ban.split())
    return ViTokenizer.tokenize(van_ban)


if DA_CO_CACHE:
    df = pd.read_csv(TEP_CACHE).dropna(subset=["van_ban_tach_tu"])
    print("Đọc lại", len(df), "văn bản đã tách từ từ", TEP_CACHE)
else:
    if MAU_THU:
        dem = collections.Counter()
        loc = []
        for r in ban_ghi:
            khoa = (r["tap"], r["chu_de"])
            if dem[khoa] < SO_MAU_MOI_LOP:
                dem[khoa] += 1
                loc.append(r)
        ban_ghi_dung = loc
        print("CHẾ ĐỘ CHẠY THỬ: chỉ dùng", len(ban_ghi_dung), "văn bản")
    else:
        ban_ghi_dung = ban_ghi
        print("CHẾ ĐỘ ĐẦY ĐỦ:", len(ban_ghi_dung), "văn bản")

    t0 = time.time()
    tong = len(ban_ghi_dung)
    moc_bao = 10000
    for i, r in enumerate(ban_ghi_dung):
        r["van_ban_tach_tu"] = ViTokenizer.tokenize(r["van_ban"])
        if i + 1 >= moc_bao:
            print("   đã tách từ", i + 1, "/", tong, "sau", round(time.time() - t0, 1), "giây")
            moc_bao += 10000
    print("Tách từ xong sau", round(time.time() - t0, 1), "giây")

    df = pd.DataFrame([{k: r[k] for k in
                        ["tap", "chu_de", "ten_tep", "toa_soan", "so_tu", "van_ban_tach_tu"]}
                       for r in ban_ghi_dung])
    df.to_csv(TEP_CACHE, index=False, encoding="utf-8")
    print("Đã lưu", TEP_CACHE, "để lần sau khỏi phải tách lại")
    print()
    print("Ví dụ một đoạn trước và sau khi tách từ:")
    print("  Trước:", ban_ghi_dung[0]["van_ban"][:90])
    print("  Sau  :", ban_ghi_dung[0]["van_ban_tach_tu"][:110])

In [ ]:
# Chia tập. Bộ dữ liệu VNTC đã có sẵn bản chia chuẩn của tác giả nên ở đây không gọi
# train_test_split mà dùng trực tiếp cột "tap".
tr = df[df["tap"] == "train"].reset_index(drop=True)
te = df[df["tap"] == "test"].reset_index(drop=True)
nhan_lop = sorted(tr["chu_de"].unique())

print("Tập huấn luyện:", len(tr), "văn bản")
print("Tập kiểm thử  :", len(te), "văn bản")
print("Số lớp        :", len(nhan_lop))

## Phần 5. Chia tập TRƯỚC rồi mới fit TF-IDF

Đây là phần dễ sai nhất trong cả bài, xin trình bày kỹ.

LỖI: gọi fit_transform trên TOÀN BỘ dữ liệu rồi mới cắt ra thành train và test.
Đoạn mã sai này chạy trơn tru, không báo lỗi gì, và đó là lý do nó nguy hiểm.

VÌ SAO SAI: hàm fit của TfidfVectorizer không phải một phép biến đổi thuần túy.
Nó HỌC hai thứ từ dữ liệu nó nhìn thấy, là bộ từ vựng và giá trị idf của từng từ.
Nếu tập mà nó nhìn thấy bao gồm cả tập kiểm thử thì trọng số mô hình dùng đã mang
thông tin của tập kiểm thử, và điểm số đo được không còn trung thực.

CÁCH ĐÚNG: chia tập trước, sau đó gọi fit_transform trên tập huấn luyện và chỉ gọi
transform trên tập kiểm thử.

Lưu ý thêm: lỗi này không nằm ở tên hàm train_test_split mà nằm ở THỨ TỰ giữa việc
chia dữ liệu và việc học tham số tiền xử lý. Notebook này không hề gọi
train_test_split, nhưng nếu ai đó nối hai tập lại rồi gọi fit_transform trên bảng
gộp cho tiện thì lỗi tái xuất hiện y nguyên, và còn khó phát hiện hơn.

In [ ]:
# MINH HỌA BẰNG SỐ: chênh lệch idf giữa cách làm đúng và cách làm sai.
#
# Công thức idf mà thư viện scikit-learn dùng (smooth_idf = True):
#     idf(t) = ln( (1 + N) / (1 + df(t)) ) + 1

def idf_sklearn(N, df_t):
    return math.log((1 + N) / (1 + df_t)) + 1

# Giả sử có 5 văn bản, 4 dùng để huấn luyện và 1 dùng để kiểm thử.
# Từ "cổ_phiếu" xuất hiện ở 1 văn bản huấn luyện và ở văn bản kiểm thử.
print("Trường hợp 1: từ có mặt ở cả hai tập")
print("   Làm ĐÚNG (fit chỉ trên train, N = 4, df = 1): idf =", round(idf_sklearn(4, 1), 4))
print("   Làm SAI  (fit trên cả 5 văn bản, N = 5, df = 2): idf =", round(idf_sklearn(5, 2), 4))
print("   Chênh lệch:", round(idf_sklearn(4, 1) - idf_sklearn(5, 2), 4))
print()

print("Trường hợp 2: từ chỉ xuất hiện trong tập kiểm thử")
print("   Làm ĐÚNG: từ này không nằm trong bộ từ vựng, lúc transform bị bỏ qua.")
print("             Đây đúng là điều sẽ xảy ra khi hệ thống chạy thật.")
print("   Làm SAI : từ này có hẳn một cột riêng với idf =", round(idf_sklearn(5, 1), 4))
print("             Trọng số cao vì nó hiếm. Nếu nó tình cờ tương quan với nhãn,")
print("             mô hình sẽ khai thác một tín hiệu lẽ ra nó không được phép biết.")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

if CHAY_CO_DIEN:
    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),      # unigram và bigram
        min_df=3,                # bỏ từ chỉ xuất hiện dưới 3 văn bản
        max_df=0.9,              # bỏ từ xuất hiện ở trên 90 phần trăm số văn bản
        sublinear_tf=True,       # thay tf bằng 1 + ln(tf)
        max_features=300000,     # chặn trần bộ nhớ
    )

    t0 = time.time()
    # THỨ TỰ BẮT BUỘC: fit_transform trên TRAIN trước
    X_train = vectorizer.fit_transform(tr["van_ban_tach_tu"].tolist())
    # rồi chỉ transform trên TEST, tuyệt đối không fit lại
    X_test = vectorizer.transform(te["van_ban_tach_tu"].tolist())
    y_train = tr["chu_de"].tolist()
    y_test = te["chu_de"].tolist()

    print("TF-IDF xong sau", round(time.time() - t0, 1), "giây")
    print("Kích thước ma trận train:", X_train.shape)
    print("Kích thước ma trận test :", X_test.shape)
    print("Số đặc trưng học được từ tập train:", len(vectorizer.vocabulary_))
else:
    y_train = tr["chu_de"].tolist()
    y_test = te["chu_de"].tolist()
    print("Bỏ qua cell này vì CHAY_CO_DIEN = False")

## Phần 6. Huấn luyện và so sánh Naive Bayes, Logistic Regression, SVM

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix


def khoang_wilson(so_dung, tong, z=1.96):
    """Khoảng tin cậy Wilson cho tỷ lệ. Ổn định hơn công thức chuẩn thông thường
    khi tỷ lệ gần 0 hoặc gần 1, và khi số mẫu nhỏ."""
    if tong == 0:
        return (0.0, 0.0)
    p = so_dung / tong
    mau_so = 1 + z * z / tong
    tam = p + z * z / (2 * tong)
    lech = z * math.sqrt(p * (1 - p) / tong + z * z / (4 * tong * tong))
    return ((tam - lech) / mau_so, (tam + lech) / mau_so)


def do_ket_qua(ten, y_that, y_doan, giay=None):
    acc = accuracy_score(y_that, y_doan)
    thap, cao = khoang_wilson(int(round(acc * len(y_that))), len(y_that))
    r = {
        "accuracy": round(float(acc), 4),
        "wilson_thap": round(thap, 4),
        "wilson_cao": round(cao, 4),
        "macro_f1": round(float(f1_score(y_that, y_doan, average="macro")), 4),
        "weighted_f1": round(float(f1_score(y_that, y_doan, average="weighted")), 4),
    }
    if giay is not None:
        r["giay_huan_luyen"] = round(giay, 1)
    print(ten)
    print("   accuracy    :", r["accuracy"],
          "(Wilson 95:", r["wilson_thap"], "đến", r["wilson_cao"], ")")
    print("   macro-F1    :", r["macro_f1"])
    print("   weighted-F1 :", r["weighted_f1"])
    if giay is not None:
        print("   thời gian   :", r["giay_huan_luyen"], "giây")
    print()
    return r


# Ba mô hình đại diện cho ba nguyên lý học khác nhau:
#   M1 hướng sinh kèm giả định độc lập
#   M2 hướng phân biệt theo xác suất
#   M3 hướng tối đa hóa lề
# Riêng M1 không đặt class_weight vì xác suất tiên nghiệm của Naive Bayes vốn đã
# phản ánh phân bố lớp, can thiệp vào sẽ phá vỡ ý nghĩa xác suất của mô hình.
CAC_MO_HINH = [
    ("M1_NaiveBayes", MultinomialNB(alpha=0.1)),
    ("M2_LogisticRegression", LogisticRegression(max_iter=1000, C=10.0, class_weight="balanced")),
    ("M3_LinearSVM", LinearSVC(C=1.0, class_weight="balanced")),
]

ket_qua = {}
du_doan_luu = {}
mo_hinh_luu = {}

if CHAY_CO_DIEN:
    for ten, mo_hinh in CAC_MO_HINH:
        t0 = time.time()
        mo_hinh.fit(X_train, y_train)
        giay = time.time() - t0
        y_du_doan = mo_hinh.predict(X_test)
        ket_qua[ten] = do_ket_qua(ten, y_test, y_du_doan, giay)
        du_doan_luu[ten] = y_du_doan
        mo_hinh_luu[ten] = mo_hinh
else:
    print("Bỏ qua cell này vì CHAY_CO_DIEN = False")

## Phần 7. Kiểm định chéo 5 lớp

Kiểm định chéo không làm con số accuracy chính xác hơn, vì tập kiểm thử đã đủ lớn.
Cái nó bổ sung là thông tin khác về chất: nó cho biết kết quả dao động bao nhiêu khi
thay đổi tập huấn luyện.

Điểm quan trọng về kỹ thuật: vectorizer phải được fit LẠI ở TỪNG lớp gấp, không được
fit một lần bên ngoài vòng lặp. Cách an toàn nhất là gói vectorizer và bộ phân lớp
vào cùng một Pipeline, khi đó thư viện tự quản lý thứ tự giúp mình và không thể rò rỉ.

Chia lớp gấp dùng kiểu phân tầng, tức giữ nguyên tỷ lệ các lớp trong từng lớp gấp.
Với dữ liệu lệch gần 4 lần, chia ngẫu nhiên không phân tầng có thể tạo ra một lớp gấp
thiếu hẳn mẫu của lớp nhỏ.

Cảnh báo về thời gian: phần này fit lại TF-IDF 15 lần nên chạy lâu hơn Phần 6 khá nhiều.
Đặt CHAY_KIEM_DINH_CHEO = False ở Phần 0 nếu muốn bỏ qua.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline

ket_qua_cv = {}

if CHAY_CO_DIEN and CHAY_KIEM_DINH_CHEO:
    # Kiểm định chéo chạy trên TẬP HUẤN LUYỆN, không đụng tới tập kiểm thử.
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=HAT_GIONG)
    # Dung .tolist() thay cho .values: tu pandas 3.x, cot kieu chuoi duoc
    # luu bang ArrowStringArray. Kieu nay khong ho tro cach lay phan tu
    # bang mang chi so ma cross_validate dung ben trong de chia lop gap,
    # nen se bao loi TypeError. List Python thuan thi luon an toan.
    X_van_ban = tr["van_ban_tach_tu"].tolist()
    y_nhan = tr["chu_de"].tolist()

    for ten, mo_hinh in CAC_MO_HINH:
        # Pipeline bảo đảm vectorizer được fit lại ở từng lớp gấp
        duong_ong = Pipeline([
            ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.9,
                                      sublinear_tf=True, max_features=300000)),
            ("mo_hinh", mo_hinh),
        ])
        t0 = time.time()
        diem = cross_validate(duong_ong, X_van_ban, y_nhan, cv=skf,
                              scoring=["accuracy", "f1_macro"], n_jobs=1)
        a = diem["test_accuracy"]
        f = diem["test_f1_macro"]
        ket_qua_cv[ten] = {
            "acc_tb": round(float(a.mean()), 4), "acc_lech": round(float(a.std(ddof=1)), 4),
            "f1_tb": round(float(f.mean()), 4), "f1_lech": round(float(f.std(ddof=1)), 4),
        }
        print(ten, "(", round(time.time() - t0, 1), "giây )")
        print("   accuracy trung bình:", ket_qua_cv[ten]["acc_tb"],
              "độ lệch chuẩn", ket_qua_cv[ten]["acc_lech"])
        print("   macro-F1 trung bình:", ket_qua_cv[ten]["f1_tb"],
              "độ lệch chuẩn", ket_qua_cv[ten]["f1_lech"])
        print()
else:
    print("Bỏ qua cell này")

## Phần 8. Ma trận nhầm lẫn và báo cáo chi tiết từng lớp

In [ ]:
def ve_ma_tran_nham_lan(y_that, y_doan, ten_mo_hinh, ten_tep):
    cm = confusion_matrix(y_that, y_doan, labels=nhan_lop)
    ve = [TEN_HIEN_THI.get(n, n) for n in nhan_lop]

    fig, ax = plt.subplots(figsize=(9, 8))
    hinh = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    ax.set_xticks(np.arange(len(nhan_lop)))
    ax.set_yticks(np.arange(len(nhan_lop)))
    ax.set_xticklabels(ve, rotation=45, ha="right")
    ax.set_yticklabels(ve)
    ax.set_xlabel("Nhãn dự đoán")
    ax.set_ylabel("Nhãn thật")
    ax.set_title("Ma trận nhầm lẫn _ " + ten_mo_hinh)
    nguong = cm.max() / 2
    for i in range(len(nhan_lop)):
        for j in range(len(nhan_lop)):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > nguong else "black", fontsize=8)
    fig.colorbar(hinh)
    plt.tight_layout()
    plt.savefig(ten_tep + ".png", dpi=150, bbox_inches="tight")
    plt.show()

    pd.DataFrame(cm, index=ve, columns=ve).to_csv(ten_tep + ".csv", encoding="utf-8")
    print("Đã lưu", ten_tep + ".png", "và", ten_tep + ".csv")
    print()

    cap = []
    for i in range(len(nhan_lop)):
        for j in range(len(nhan_lop)):
            if i != j and cm[i, j] > 0:
                cap.append((int(cm[i, j]), ve[i], ve[j]))
    cap.sort(reverse=True)
    print("MƯỜI CẶP BỊ NHẦM NHIỀU NHẤT (nhãn thật _ bị đoán thành _ số lượng)")
    for so, that, doan in cap[:10]:
        print("   ", that, "_", doan, "_", so)
    return cm


if CHAY_CO_DIEN:
    TEN_TOT_NHAT = max(ket_qua, key=lambda k: ket_qua[k]["macro_f1"])
    print("Mô hình cổ điển có macro-F1 cao nhất:", TEN_TOT_NHAT)
    print()
    print("BÁO CÁO CHI TIẾT TỪNG LỚP")
    print(classification_report(y_test, du_doan_luu[TEN_TOT_NHAT],
                                target_names=nhan_lop, digits=4, zero_division=0))
    ve_ma_tran_nham_lan(y_test, du_doan_luu[TEN_TOT_NHAT], TEN_TOT_NHAT, "ma_tran_nham_lan_co_dien")
else:
    TEN_TOT_NHAT = None
    print("Bỏ qua cell này vì CHAY_CO_DIEN = False")

## Phần 9. Tinh chỉnh PhoBERT

CẢNH BÁO TRUNG THỰC: phần này nhóm em CHƯA chạy thử được, vì máy dùng để soạn
notebook không có GPU và không tải được mô hình từ kho Hugging Face. Code được viết
theo tài liệu chính thức của thư viện transformers và của PhoBERT, nhưng chưa qua
thực nghiệm. Nếu chạy báo lỗi, hãy gửi lại nguyên văn thông báo lỗi.

Ba điểm kỹ thuật của phần này:

Một, PhoBERT giới hạn cứng 256 token con cho mỗi văn bản. Trong khi đó văn bản trong
kho ngữ liệu có độ dài trung vị 361 từ, và một từ tiếng Việt thường tách ra hơn một
token con. Nghĩa là phần lớn văn bản sẽ bị cắt mất quá nửa nội dung. Đây là ràng buộc
kiến trúc, không né được.

Hai, batch size 16 kèm gộp đạo hàm 2 cho batch hiệu dụng 32. Cách này giữ nguyên động
lực học của cấu hình chuẩn mà không tràn bộ nhớ trên GPU 8 GB.

Ba, mô hình được lưu sau mỗi epoch. Nếu phiên chạy đứt giữa chừng thì không mất trắng.

In [ ]:
# Đo số token con trung bình trên một từ. Con số này quyết định thực tế
# giữ lại được bao nhiêu từ trong giới hạn 256 token con.
if CHAY_PHOBERT:
    from transformers import AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(MO_HINH_GOC)
    print("Đã nạp bộ tách token của", MO_HINH_GOC)
    print("Kích thước từ vựng:", tokenizer.vocab_size)
    print()

    mau_do = tr["van_ban_tach_tu"].sample(n=min(2000, len(tr)), random_state=HAT_GIONG)
    tong_tu = 0
    tong_token = 0
    for vb in mau_do:
        tong_tu += len(str(vb).split())
        tong_token += len(tokenizer.encode(str(vb), add_special_tokens=False))
    ty_le_r = tong_token / max(tong_tu, 1)

    print("Đo trên", len(mau_do), "văn bản của tập huấn luyện:")
    print("   Số token con trung bình trên một từ (r) =", round(ty_le_r, 4))
    print("   Số vị trí dành cho nội dung = 256 - 2 =", DO_DAI_TOI_DA - 2)
    print("   Số từ giữ lại được trung bình =", round((DO_DAI_TOI_DA - 2) / ty_le_r, 1))
    print("   Trong khi độ dài trung vị của văn bản là 361 từ.")
    print()
    print("Ví dụ tách token một câu ngắn:")
    cau = "Đội_tuyển Việt_Nam giành chiến_thắng"
    print("   ", cau)
    print("   ", tokenizer.tokenize(cau))
else:
    ty_le_r = None
    print("Bỏ qua cell này vì CHAY_PHOBERT = False")

In [ ]:
# Mã hóa văn bản thành dãy chỉ số token con, kèm chiến lược cắt.
if CHAY_PHOBERT:
    import torch
    from torch.utils.data import DataLoader, TensorDataset

    torch.manual_seed(HAT_GIONG)
    THIET_BI = "cuda" if torch.cuda.is_available() else "cpu"

    nhan_to_so = {n: i for i, n in enumerate(nhan_lop)}
    ID_DAU = tokenizer.cls_token_id
    ID_CUOI = tokenizer.sep_token_id
    ID_DEM = tokenizer.pad_token_id


    def cat_chuoi(ids, kieu):
        """head lấy phần đầu. head_tail lấy nửa đầu cộng nửa cuối.
        Chủ đề tin tức thường bộc lộ ngay ở tiêu đề và đoạn dẫn nên head là
        cấu hình chính; head_tail dùng làm thí nghiệm phụ để so sánh."""
        con = DO_DAI_TOI_DA - 2
        if len(ids) <= con:
            giu = ids
        elif kieu == "head_tail":
            nua = con // 2
            giu = ids[:nua] + ids[len(ids) - (con - nua):]
        else:
            giu = ids[:con]
        return [ID_DAU] + giu + [ID_CUOI]


    def ma_hoa(khung, kieu):
        n = len(khung)
        X = np.full((n, DO_DAI_TOI_DA), ID_DEM, dtype=np.int64)
        M = np.zeros((n, DO_DAI_TOI_DA), dtype=np.int64)
        moc = 10000
        for i, vb in enumerate(khung["van_ban_tach_tu"].tolist()):
            ids = tokenizer.encode(str(vb), add_special_tokens=False)
            ids = cat_chuoi(ids, kieu)
            X[i, :len(ids)] = ids
            M[i, :len(ids)] = 1
            if i + 1 >= moc:
                print("   mã hóa", i + 1, "/", n)
                moc += 10000
        y = np.array([nhan_to_so[c] for c in khung["chu_de"].tolist()], dtype=np.int64)
        return TensorDataset(torch.tensor(X), torch.tensor(M), torch.tensor(y))


    print("Mã hóa tập huấn luyện")
    ds_train = ma_hoa(tr, KIEU_CAT)
    print("Mã hóa tập kiểm thử")
    ds_test = ma_hoa(te, KIEU_CAT)

    dl_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True)
    dl_test = DataLoader(ds_test, batch_size=BATCH_SIZE * 2, shuffle=False)
    print("Xong. Số batch huấn luyện:", len(dl_train), "| số batch kiểm thử:", len(dl_test))
else:
    print("Bỏ qua cell này vì CHAY_PHOBERT = False")

In [ ]:
# Vòng huấn luyện. Đây là cell chạy lâu nhất của cả notebook.
if CHAY_PHOBERT:
    from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup

    mo_hinh_pho = AutoModelForSequenceClassification.from_pretrained(
        MO_HINH_GOC, num_labels=len(nhan_lop)).to(THIET_BI)

    opt = torch.optim.AdamW(mo_hinh_pho.parameters(), lr=TOC_DO_HOC, weight_decay=0.01)
    so_buoc_moi_epoch = math.ceil(len(dl_train) / GOP_DAO_HAM)
    tong_buoc = so_buoc_moi_epoch * SO_EPOCH
    lich = get_linear_schedule_with_warmup(opt, int(0.1 * tong_buoc), tong_buoc)
    dung_fp16 = (THIET_BI == "cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=dung_fp16)

    print("Bắt đầu tinh chỉnh")
    print("   Thiết bị          :", THIET_BI)
    print("   Batch size        :", BATCH_SIZE, "| gộp đạo hàm", GOP_DAO_HAM,
          "| batch hiệu dụng", BATCH_SIZE * GOP_DAO_HAM)
    print("   Số epoch          :", SO_EPOCH)
    print("   Tổng số bước cập nhật:", tong_buoc)
    print()

    t_bat_dau = time.time()
    for epoch in range(SO_EPOCH):
        mo_hinh_pho.train()
        tong_loss = 0.0
        opt.zero_grad(set_to_none=True)
        dem_gop = 0
        moc_in = 200
        for buoc, (x, m, y) in enumerate(dl_train):
            x, m, y = x.to(THIET_BI), m.to(THIET_BI), y.to(THIET_BI)
            with torch.amp.autocast("cuda", dtype=torch.float16, enabled=dung_fp16):
                ra = mo_hinh_pho(input_ids=x, attention_mask=m, labels=y)
                loss = ra.loss / GOP_DAO_HAM
            scaler.scale(loss).backward()
            tong_loss += ra.loss.item()

            # Chỉ cập nhật trọng số sau mỗi GOP_DAO_HAM batch
            dem_gop += 1
            if dem_gop >= GOP_DAO_HAM or (buoc + 1) == len(dl_train):
                dem_gop = 0
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(mo_hinh_pho.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
                lich.step()
                opt.zero_grad(set_to_none=True)

            if buoc + 1 >= moc_in:
                print("   epoch", epoch + 1, "batch", buoc + 1, "/", len(dl_train),
                      "| loss trung bình", round(tong_loss / (buoc + 1), 4),
                      "|", round(time.time() - t_bat_dau, 1), "giây")
                moc_in += 200

        print("Xong epoch", epoch + 1, "| loss trung bình", round(tong_loss / len(dl_train), 4))
        # Lưu điểm dừng sau mỗi epoch để nếu đứt phiên thì không mất trắng
        mo_hinh_pho.save_pretrained("phobert_epoch_" + str(epoch + 1))
        tokenizer.save_pretrained("phobert_epoch_" + str(epoch + 1))
        print("   Đã lưu điểm dừng phobert_epoch_" + str(epoch + 1))
        print()

    giay_phobert = time.time() - t_bat_dau
    print("Tinh chỉnh xong sau", round(giay_phobert / 60, 1), "phút")
else:
    giay_phobert = None
    print("Bỏ qua cell này vì CHAY_PHOBERT = False")

In [ ]:
# Đánh giá PhoBERT trên cùng tập kiểm thử với nhánh cổ điển
if CHAY_PHOBERT:
    mo_hinh_pho.eval()
    gom = []
    with torch.no_grad():
        for x, m, y in dl_test:
            x, m = x.to(THIET_BI), m.to(THIET_BI)
            with torch.amp.autocast("cuda", dtype=torch.float16, enabled=dung_fp16):
                logits = mo_hinh_pho(input_ids=x, attention_mask=m).logits
            gom.append(logits.float().argmax(dim=-1).cpu().numpy())

    y_doan_so = np.concatenate(gom)
    y_doan_pho = np.array([nhan_lop[i] for i in y_doan_so])

    TEN_PHOBERT = "M4_PhoBERT_" + KIEU_CAT
    ket_qua[TEN_PHOBERT] = do_ket_qua(TEN_PHOBERT, y_test, y_doan_pho, giay_phobert)
    du_doan_luu[TEN_PHOBERT] = y_doan_pho

    print("BÁO CÁO CHI TIẾT TỪNG LỚP")
    print(classification_report(y_test, y_doan_pho, target_names=nhan_lop,
                                digits=4, zero_division=0))
    ve_ma_tran_nham_lan(y_test, y_doan_pho, TEN_PHOBERT, "ma_tran_nham_lan_phobert")
else:
    TEN_PHOBERT = None
    print("Bỏ qua cell này vì CHAY_PHOBERT = False")

## Phần 10. Hàm dự đoán cho văn bản mới

In [ ]:
import joblib

VAN_BAN_THU = [
    "Đội tuyển Việt Nam giành chiến thắng 2-0 trong trận đấu tối qua tại sân Mỹ Đình, tiền đạo ghi bàn ở phút 78.",
    "Giá cổ phiếu ngân hàng đồng loạt tăng trần, chỉ số VN-Index vượt mốc 1.200 điểm trong phiên giao dịch hôm nay.",
    "Các nhà khoa học vừa công bố phát hiện mới về cấu trúc protein giúp hiểu rõ hơn cơ chế phân chia tế bào.",
    "Bộ Y tế khuyến cáo người dân tiêm nhắc lại vắc xin và rửa tay thường xuyên để phòng bệnh hô hấp mùa lạnh.",
    "Bộ vi xử lý thế hệ mới hỗ trợ bộ nhớ DDR5 và có hiệu năng đồ họa tích hợp cao hơn đời trước.",
]

if CHAY_CO_DIEN:
    # Dùng Logistic Regression cho hàm dự đoán vì mô hình này cho xác suất thật sự.
    # LinearSVC chỉ cho khoảng cách tới mặt phân chia chứ không phải xác suất, nên nếu
    # ép nó thành xác suất thì con số đó không có ý nghĩa hiệu chuẩn.
    mo_hinh_trien_khai = mo_hinh_luu["M2_LogisticRegression"]

    # Lưu cả vectorizer và mô hình vào cùng một tệp.
    # Điểm cốt lõi: lúc dự đoán phải NẠP LẠI đúng vectorizer đã được fit lúc huấn luyện,
    # tuyệt đối không fit lại trên dữ liệu người dùng gửi lên. Nếu fit lại thì bộ từ vựng
    # và toàn bộ trọng số idf sẽ khác đi, mô hình nhận vào một không gian đặc trưng xa lạ
    # và kết quả dự đoán sẽ vô nghĩa.
    joblib.dump({"vectorizer": vectorizer, "mo_hinh": mo_hinh_trien_khai,
                 "nhan_lop": list(mo_hinh_trien_khai.classes_)}, "mo_hinh_phan_loai.joblib")
    print("Đã lưu mo_hinh_phan_loai.joblib")
    print()

    def du_doan_co_dien(van_ban, top_k=3):
        """Nhận văn bản tiếng Việt thô, trả về nhãn chủ đề kèm độ tin cậy.
        Văn bản phải đi qua ĐÚNG chuỗi tiền xử lý đã dùng lúc huấn luyện; nếu bỏ sót
        bước tách từ thì độ chính xác sẽ sụt mạnh mà không báo lỗi gì."""
        x = vectorizer.transform([tien_xu_ly(van_ban)])
        p = mo_hinh_trien_khai.predict_proba(x)[0]
        thu_tu = np.argsort(p)[::-1][:top_k]
        return [(mo_hinh_trien_khai.classes_[i], float(p[i])) for i in thu_tu]

    print("DỰ ĐOÁN BẰNG MÔ HÌNH CỔ ĐIỂN")
    for vb in VAN_BAN_THU:
        print("Văn bản:", vb[:70])
        for nhan, p in du_doan_co_dien(vb):
            print("     ", TEN_HIEN_THI.get(nhan, nhan), "_ độ tin cậy", round(p, 4))
        print()

In [ ]:
if CHAY_PHOBERT:
    def du_doan_phobert(van_ban, top_k=3):
        """Dự đoán bằng PhoBERT. Văn bản cũng phải đi qua đúng chuỗi tiền xử lý
        như lúc huấn luyện, đặc biệt là bước tách từ."""
        ids = tokenizer.encode(tien_xu_ly(van_ban), add_special_tokens=False)
        ids = cat_chuoi(ids, KIEU_CAT)
        x = torch.tensor([ids + [ID_DEM] * (DO_DAI_TOI_DA - len(ids))]).to(THIET_BI)
        m = torch.tensor([[1] * len(ids) + [0] * (DO_DAI_TOI_DA - len(ids))]).to(THIET_BI)
        mo_hinh_pho.eval()
        with torch.no_grad():
            logits = mo_hinh_pho(input_ids=x, attention_mask=m).logits[0]
        p = torch.softmax(logits.float(), dim=-1).cpu().numpy()
        thu_tu = np.argsort(p)[::-1][:top_k]
        return [(nhan_lop[i], float(p[i])) for i in thu_tu]

    print("DỰ ĐOÁN BẰNG PhoBERT")
    for vb in VAN_BAN_THU:
        print("Văn bản:", vb[:70])
        for nhan, p in du_doan_phobert(vb):
            print("     ", TEN_HIEN_THI.get(nhan, nhan), "_ độ tin cậy", round(p, 4))
        print()
else:
    print("Bỏ qua cell này vì CHAY_PHOBERT = False")

## Phần 11. Bảng tổng hợp kết quả

Cell dưới đây in ra bảng tổng hợp để copy gửi lại. Mọi con số trong bảng đều là kết
quả chạy thật của notebook này, không có giá trị nào được điền tay.

Kết quả cũng được lưu ra tệp bang_tong_hop.json. Nếu chạy hai lượt riêng cho hai
nhánh thì tệp này được gộp lại chứ không bị ghi đè.

In [ ]:
# Gộp với kết quả của lượt chạy trước nếu có, để chạy hai lượt riêng vẫn ra một bảng đủ
TEP_TONG_HOP = "bang_tong_hop.json"
tong_hop = {}
if os.path.exists(TEP_TONG_HOP):
    with open(TEP_TONG_HOP, encoding="utf-8") as f:
        tong_hop = json.load(f)

tong_hop.setdefault("du_lieu", tk_trung_lap)
tong_hop["che_do_chay_thu"] = MAU_THU
tong_hop["so_train"] = len(tr)
tong_hop["so_test"] = len(te)
tong_hop["so_lop"] = len(nhan_lop)
if CHAY_CO_DIEN:
    tong_hop["so_dac_trung_tfidf"] = int(X_train.shape[1])
if ty_le_r is not None:
    tong_hop["token_con_tren_mot_tu"] = round(float(ty_le_r), 4)
tong_hop.setdefault("ket_qua_test", {}).update(ket_qua)
tong_hop.setdefault("ket_qua_cv", {}).update(ket_qua_cv)

with open(TEP_TONG_HOP, "w", encoding="utf-8") as f:
    json.dump(tong_hop, f, ensure_ascii=False, indent=1)

kq_all = tong_hop["ket_qua_test"]
cv_all = tong_hop["ket_qua_cv"]

print("=" * 78)
print("BẢNG TỔNG HỢP KẾT QUẢ")
print("=" * 78)
print()
print("A. DỮ LIỆU")
print("   Kho ngữ liệu       : VNTC 10Topics Ver1.1, github.com/duyvuleo/VNTC")
for k, v in tong_hop.get("du_lieu", {}).items():
    print("   {:<19}:".format(k), v)
print("   Dùng thực tế       :", tong_hop["so_train"], "train +", tong_hop["so_test"],
      "test =", tong_hop["so_train"] + tong_hop["so_test"])
print("   Số lớp             :", tong_hop["so_lop"])
print("   Chế độ             :", "CHẠY THỬ (một phần dữ liệu)" if MAU_THU else "ĐẦY ĐỦ")
if "so_dac_trung_tfidf" in tong_hop:
    print("   Số đặc trưng TF-IDF:", tong_hop["so_dac_trung_tfidf"])
if "token_con_tren_mot_tu" in tong_hop:
    print("   Token con trên một từ:", tong_hop["token_con_tren_mot_tu"])
print()

print("B. KẾT QUẢ TRÊN TẬP KIỂM THỬ")
print()
dong = "{:<24}{:>10}{:>22}{:>11}{:>13}{:>9}"
print(dong.format("Mô hình", "Accuracy", "Wilson 95", "macro-F1", "weighted-F1", "Giây"))
print("-" * 89)
for ten in sorted(kq_all):
    r = kq_all[ten]
    print(dong.format(ten, r["accuracy"],
                      str(r["wilson_thap"]) + " _ " + str(r["wilson_cao"]),
                      r["macro_f1"], r["weighted_f1"], r.get("giay_huan_luyen", "")))
print()

if cv_all:
    print("C. KIỂM ĐỊNH CHÉO 5 LỚP (chạy trên tập huấn luyện)")
    print()
    dong2 = "{:<24}{:>22}{:>24}"
    print(dong2.format("Mô hình", "Accuracy trung bình", "macro-F1 trung bình"))
    print("-" * 70)
    for ten in sorted(cv_all):
        r = cv_all[ten]
        print(dong2.format(ten,
                           str(r["acc_tb"]) + " +/- " + str(r["acc_lech"]),
                           str(r["f1_tb"]) + " +/- " + str(r["f1_lech"])))
    print()

if kq_all:
    tot_nhat = max(kq_all, key=lambda k: kq_all[k]["macro_f1"])
    print("D. MÔ HÌNH TỐT NHẤT THEO macro-F1:", tot_nhat)
    print("   macro-F1 =", kq_all[tot_nhat]["macro_f1"])
    print()

    if tot_nhat in du_doan_luu:
        print("E. F1 TỪNG LỚP CỦA MÔ HÌNH TỐT NHẤT")
        bc = classification_report(y_test, du_doan_luu[tot_nhat], target_names=nhan_lop,
                                   output_dict=True, zero_division=0)
        for n in nhan_lop:
            print("   {:<20} F1 = {:.4f}   số mẫu = {}".format(
                TEN_HIEN_THI.get(n, n), bc[n]["f1-score"], int(bc[n]["support"])))
        print()

print("=" * 78)
print("Đã lưu", TEP_TONG_HOP)
print()
print("CÁC TỆP ĐÃ SINH RA:")
for t in sorted(os.listdir(".")):
    if t.endswith((".png", ".csv", ".json", ".joblib")):
        print("   ", t, "_", round(os.path.getsize(t) / 1024, 1), "KB")